# LLM Process

A notebook used to simplify (ie. remove columns from & restructure) the dataset for LLM processing

In [41]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 

## Setup

In [70]:
data_dir = "../raw"
filename = data_dir + "/merged_2022_2025.csv"
raw_df = df = pd.read_csv(filename, index_col=0, header=[0, 1], low_memory=False)

In [71]:
flat = raw_df.copy()
flat.columns = [f"{a.replace(' ', "_")}_{b.replace(' ', "_")}" for a, b in flat.columns]
flat.head()

,Time_Date,Time_Local_Time_Of_Day,Place_Locale_Reference,Place_State_Reference,Place_Relative_Position.Angle.Radial,Place_Relative_Position.Distance.Nautical_Miles,Place_Altitude.AGL.Single_Value,Place_Altitude.MSL.Single_Value,Place_Latitude_/_Longitude_(UAS),Environment_Flight_Conditions,...,Events_Detector,Events_When_Detected,Events_Result,Assessments_Contributing_Factors_/_Situations,Assessments_Primary_Problem,Report_1_Narrative,Report_1_Callback,Report_2_Narrative,Report_2_Callback,Report_1_Synopsis
2260174,202507,1201-1800,BWI.Airport,MD,NaN,NaN,NaN,900.0,NaN,VMC,...,Person Air Traffic Control,In-flight,General None Reported / Taken,Procedure; Human Factors; ATC Equipment / Nav ...,Ambiguous,Aircraft vectored in within 1NM to final appro...,NaN,NaN,NaN,Air carrier Captain reported a low altitude al...
2260249,202507,0601-1200,ZZZ.Airport,US,NaN,NaN,NaN,NaN,NaN,NaN,...,Automation Aircraft Other Automation; Person F...,In-flight,Flight Crew Executed Go Around / Missed Approa...,Human Factors; Aircraft,Ambiguous,While on short final we received a glideslope ...,NaN,NaN,NaN,Air carrier pilot reported receiving an aircra...
2260370,202507,1801-2400,ZZZ.ARTCC,US,NaN,NaN,NaN,35000.0,NaN,VMC,...,Person Flight Crew,In-flight,Air Traffic Control Issued New Clearance; Air ...,Aircraft,Aircraft,Flying at cruise; FL350; the FO was the PF and...,NaN,At cruise; FL350; during level-off; the Captai...,NaN,B737 flight crew reported observing a slowing ...
2261277,202507,1801-2400,ZZZ.Airport,US,NaN,2.6,160.0,NaN,NaN,NaN,...,Person Flight Crew,In-flight,General None Reported / Taken,Human Factors,Human Factors,On Day 0 around XA:30; I forgot to get LAANC a...,Reporter stated TRUST certificate was obtained...,NaN,NaN,Recreational / Hobbyist UAS pilot reported fly...
2261317,202507,NaN,ZZZ.Airport,US,NaN,NaN,NaN,9000.0,NaN,VMC,...,Automation Aircraft Other Automation; Person F...,In-flight,Flight Crew Became Reoriented; Flight Crew Reg...,Weather; Human Factors; Procedure,Procedure,Divert into ZZZ from ZZZ1. FO flying. Vectors ...,NaN,Extremely strong winds blew us off the LOC whi...,NaN,B737 flight crew reported the pilot flying in ...


In [72]:
narrative_only = flat[
    ["Assessments_Primary_Problem", "Report_1_Narrative", "Report_2_Narrative"]
]

### Cleaning Target & De-duplication
Here we examine if the target is NaN or dataset has any duplicates. Both Report_1_Narrative & Report_2_Narrative have duplicates (which shouldn't be the case).

In [78]:
narrative_only.info()

<class 'pandas.DataFrame'>
Index: 20963 entries, 2260174 to 2315041
Data columns (total 3 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   Assessments_Primary_Problem  20542 non-null  str  
 1   Report_1_Narrative           20963 non-null  str  
 2   Report_2_Narrative           4493 non-null   str  
dtypes: str(3)
memory usage: 655.1 KB


In [73]:
narrative_only.describe()

,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative
count,20542,20963,4493
unique,97,20959,4342
top,Aircraft,The UAS lost the telemetry link 1;300 feet fro...,[Report narrative contained no additional info...
freq,6897,3,103


In [ ]:
# Identify Report 1 Narrative that are duplicated

r1_dupes = narrative_only[
    narrative_only.Report_1_Narrative.duplicated(keep=False)
]
r1_dupes.dropna(how="all")

,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative
2291690,Aircraft,On approach into ZZZ runway XXR; landing gear ...,After an otherwise uneventful flight; while on...
2014961,Aircraft,The UAS lost the telemetry link 1;300 feet fro...,NaN
2014962,Aircraft,The UAS lost the telemetry link 1;300 feet fro...,NaN
2014963,Aircraft,The UAS lost the telemetry link 1;300 feet fro...,NaN
2233790,Human Factors,In attempt to do my short field landing coming...,NaN
2242665,Ambiguous,In attempt to do my short field landing coming...,NaN
2303286,Aircraft,On approach into ZZZ runway XXR; landing gear ...,After an otherwise uneventful flight; while on...


In [86]:
r2_only = narrative_only.copy()
r2_dupes = r2_only[r2_only.Report_2_Narrative.duplicated(keep=False) & pd.notna(r2_only.Report_2_Narrative)]
r2_dupes.dropna(how="all")
display(r2_dupes.Report_2_Narrative.value_counts())
print("Sentinel values have lots of duplicates.")


Report_2_Narrative
[Report narrative contained no additional information.]                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

Sentinel values have lots of duplicates.


In [99]:
# Find all occurrences of sentinel value string to ensure we don't delete actual data
# r2_no_sentinel = []
sentinel = "contained no additional information"
print("R1 Narrative w/ sentinels")
display(narrative_only[
    narrative_only.Report_1_Narrative.str.contains(sentinel)
])
print("R2 Narrative w/ sentinels")
display(narrative_only[narrative_only.Report_2_Narrative.str.contains(sentinel)][
    ["Report_2_Narrative"]
].value_counts())
print("So we can safely replace sentinel string w/ NaN & not delete real info.")

R1 Narrative w/ sentinels


,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative


R2 Narrative w/ sentinels


Report_2_Narrative                                     
[Report narrative contained no additional information.]    103
[Narrative contained no additional information.]            22
[Report narrative contained no additional information]       7
[Report narrative contained no additional information].      3
[Narrative contained no additional information]              2
[Report contained no additional information.]                2
Name: count, dtype: int64

So we can safely replace sentinel string w/ NaN & not delete real info.


In [ ]:
no_nan_r2_dupes = r2_dupes[~r2_dupes.Report_2_Narrative.str.contains(sentinel)]
no_nan_r2_dupes.sort_values(by="Report_2_Narrative")

,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative
2298894,Ambiguous,On Day 0 at XA30 while landing at ZZZ in Aircr...,390 NM North of ZZZ we received a TR Unit ligh...
1986551,Aircraft,FL380 TR (Transformer/Rectifier) unit light il...,390 NM North of ZZZ we received a TR Unit ligh...
1980314,Aircraft,On climb out received SSCU (Spoiler Stabilizer...,After departure on climb received SSCU (Spoile...
2292528,Aircraft; Ambiguous,On climb out received SSCU (Spoiler Stabilizer...,After departure on climb received SSCU (Spoile...
2306821,Human Factors,It was a quiet day at the untowered airport du...,During pushback from gate the ramp crew pushed...
1995187,Human Factors,During push from gate; the ground crew pushed ...,During pushback from gate the ramp crew pushed...
2292509,Human Factors,After all passengers boarded; the Gate Agent (...,During the approach into BUR we were level at ...
1980310,Aircraft,As we were descending on the arrival (JANNY5);...,During the approach into BUR we were level at ...
2288344,Human Factors,Takeoff from Rwy XX at ZZZ. Advised the preced...,I was in the back seat and the pilot flying bo...
1976420,Aircraft,Landing Runway XX with 10 kt. wind from 180 de...,I was in the back seat and the pilot flying bo...


In [ ]:
# Searching these indices on the DB, they indeed do NOT have a primary problem assigned.
prob_null = narrative_only[pd.isna(narrative_only.Assessments_Primary_Problem)]
prob_null

,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative
1870580,NaN,Departed ZZZ at approx XA30 PST for ZZZ1. Weat...,NaN
1871747,NaN,I was flying simulated IFR (under the hood) at...,NaN
1874760,NaN,I was returning to my home airport and was awa...,NaN
1876550,NaN,Runway switch to a previously closed runway. V...,NaN
1877264,NaN,During a normal takeoff and climb from Runway ...,We were 'cleared for takeoff Runway XX full le...
...,...,...,...
2312691,NaN,On Day 0; between XA30z -XC00z; operating as P...,NaN
2314212,NaN,Exiting the runway on EWR Runway 22R at night ...,NaN
2314249,NaN,On Day 0 we were given an IFR clearance from S...,On departure after climbing to 7.000 on the he...
2314640,NaN,Sent a CPDLC route change for direct IOW (Iowa...,While in Cruise in contact with Cleveland Cent...


After referencing the DB, we can: 
1) Drop all items where the primary problem is NaN (since there is no label available)
2) Deduplicate & keep the first Report_1_Narrative (indeed, the narratives are exactly the same and in ONE case -- ACN 2233790/2242665 -- they have different primary problem labels)
3) Replace sentinel values (eg. "contained not additional info") with NaN
4) Remove all other rows where the `Report_2_Narrative` is duplicated -- sometimes, there are cases where the exported ASRS data has a report 2 narrative, but on the site, it doesn't (eg. ACN 2288344/1976420)

In [ ]:
dedupe = 